In [31]:
# imports
import os, unicodedata
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from langchain_ollama import ChatOllama

In [32]:
# modelo local no Ollama (API nativa: sem /v1 no final da URL)

MODEL = 'qwen3.5:9b'
BASE_URL = 'http://localhost:11434'
modelo = ChatOllama(model=MODEL, base_url=BASE_URL,
    temperature=0, num_predict=2560, validate_model_on_init=False
    )

print(f'Usando o modelo {MODEL} em {BASE_URL}')

Usando o modelo qwen3.5:9b em http://localhost:11434


In [ ]:
#Chave API: 396f55d7752b4d8fa275a8cda1bade64.NXMtcP8WN19baa7AnnwYifn3

In [ ]:
# Exemplo de criação de envelope com llm:
# 1. Criamos a agente Ana
ana = create_agent(
    model=modelo,
    tools=[],
    system_prompt="Você é Ana. Resuma textos de forma curta."
)

# 2. Função que cria o envelope
def criar_envelope(remetente, destinatario, conteudo):
    print (remetente, destinatario, conteudo)
    print (destinatario)
    return {
        "remetente": remetente,
        "destinatario": destinatario,
        "conteudo": conteudo
    }

# 3. Permissões do sistema
permissoes = {
    "supervisor": ["ana"]
}

# 4. Ferramenta de comunicação
@tool
def enviar_para_ana(conteudo: str) -> str:
    """Envia um texto para Ana resumir."""

    envelope = criar_envelope(
        "supervisor", "ana", conteudo
    )

    # Verificamos a autorização
    if envelope["destinatario"] not in permissoes.get(
        envelope["remetente"], []
    ):
        return "Erro: sem autorização"

    # Delegamos a tarefa à Ana
    resposta = ana.invoke({
        "messages": [
            {
                "role": "user",
                "content": envelope["conteudo"]
            }
        ]
    })

    return str(resposta["messages"][-1].content)

# 5. Criamos o supervisor
supervisor = create_agent(
    model=modelo,
    tools=[enviar_para_ana],
    system_prompt=(
        "Você é um supervisor. "
        "Sempre delegue tarefas de resumo à Ana "
        "usando a ferramenta enviar_para_ana."
    )
)

# 6. Executamos o sistema
resultado = supervisor.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Resuma: Sistemas multiagentes permitem "
                "que diferentes agentes especializados "
                "colaborem na resolução de problemas."
            )
        }
    ]
})

supervisor ana Sistemas multiagentes permitem que diferentes agentes especializados colaborem na resolução de problemas.
ana


In [ ]:
# criando um robo
import random

class Robo:
    def __init__(self, nome, bateria, localizacao):
        self.nome = nome
        self.bateria = bateria
        self.localizacao = localizacao

    def proposta(self):
        return{
            "nome": self.nome,
            "bateria": self.bateria,
            "localizacao": self.localizacao
        }

# robôs disponíveis
lamar = Robo("lamar", random.randint(0, 100), "setor "+str(random.choice(["A", "B", "C"])))
bob = Robo("bob", random.randint(0, 100), "setor "+str(random.choice(["A", "B", "C"])))
amanda = Robo("amanda", random.randint(0, 100), "setor "+str(random.choice(["A", "B", "C"])))

lista_de_robos = [lamar, bob, amanda]

# lista de distâncias entre os setores - metros
distancia_setores = {
    "setor A": {"setor A": 0, "setor B": 10, "setor C": 20},
    "setor B": {"setor B": 10, "setor B": 0, "setor C": 15},
    "setor C": {"setor C": 20, "setor B": 15, "setor C": 0}
}

def anunciar(distancia_minima, bateria_minima):
    return {"distancia_minima": distancia_minima, "bateria_minima": bateria_minima}

# ferramenta para verificar se os robôs estão aptos a realizar a tarefa no setor específico
@tool
def verificar_proposta(distancia, bateria, setor):
    """Verificar se todos os robos disponíveis para transporte de materiais
    distancia - qual a distancia minima necessaria para o transporte
    bateria - qual a bateria minima necessaria para o transporte
    setor - qual o nome do setor que está mais acessível ao robô, considere os setores A, B e C
    """
    anuncio = anunciar(distancia, bateria)

    propostas_disponiveis = []
    for robo in lista_de_robos:
        if distancia_setores[robo.localizacao][setor] <= anuncio["distancia_minima"] and robo.bateria >= anuncio["bateria_minima"]:
            propostas_disponiveis.append(robo.proposta())

        print(propostas_disponiveis)
        return propostas_disponiveis

agente_proposta = create_agent(
        model=modelo,
        tools=[verificar_proposta],
        system_prompt=(
            "Você é um robô supervisor responsável por verificar os robôs disponíveis, localização, bateria e distância do setor a ser enviado."
        ))

# agente que controla robôs e verifica se eles estão aptos a realizar tarefa no setor específico
entrada_usuario = input()
resposta_supervisor = agente_proposta.invoke({"messages": [{"role": "user", "content": entrada_usuario}]})
print(resposta_supervisor["messages"][-1].content)



In [33]:
import random

class robo:
  def __init__(self, nome, bateria, localizacao):
    self.nome = nome
    self.bateria = bateria
    self.localizacao = localizacao
    
  def proposta(self):
    return{
        "nome": self.nome,
        "bateria": self.bateria,
        "localizacao": self.localizacao
    }

lamar = robo("lamar", random.randint(0,100), "setor "+str(random.choice(["A","B","C"])))
bob = robo("bob", random.randint(0,100), "setor "+str(random.choice(["A","B","C"])))
samanta = robo("samanta", random.randint(0,100), "setor "+str(random.choice(["A","B","C"])))

todos_os_robos = [lamar, bob, samanta]

def anunciar(distancia_minima, bateria_minima):
  return{
    "distancia_minima": distancia_minima,
    "bateria_minima": bateria_minima
}

distancia_entre_setores = {
    "setor A": {
                "setor B": 10,
                "setor C": 15,
                "setor A": 0
                },
    "setor B": {
                "setor A": 10,
                "setor C": 5,
                "setor B": 0,
                },
    "setor C": {
                "setor A": 15,
                "setor B": 5,
                "setor C": 0,
                }
}

@tool
def verificar_propostas(distancia, bateria, setor):
  """Verifica todos os robos disponiveis para trasporte de materiais
  distancia: Qual a distancia minima necessaria.
  bateria: Qual a energia minima necessaria
  setor: O setor que se pretende mandar, ele aceita (setor A, setor B ou setor C), se não for for repassado um setor, retornar uma mensagem com erro de 
  - setor não informado, escolhendo um aleatório
  """

  anuncio = anunciar(distancia, bateria)

  propostas_disponiveis = []
  for robo in todos_os_robos:
    if distancia_entre_setores[robo.localizacao][setor] <= anuncio["distancia_minima"] and robo.bateria >= anuncio["bateria_minima"]:
      propostas_disponiveis.append(robo.proposta())

    print(propostas_disponiveis)
    return propostas_disponiveis

#agente que controla robos
agente_supervisor = create_agent(modelo, tools=[verificar_propostas], system_prompt="você é um coordenador que fala com se estivesse na guerra, você manda robos para setores, você define qual o valor minimo de bateria e distancia para cumprir atividades, chute uma distance e um nivel maximo de bateria")
input_do_usuario = input()
resposta_supervisor = agente_supervisor.invoke({"messages": [{"role": "user", "content": input_do_usuario}]})

print(resposta_supervisor["messages"][-1].content)

#print(resposta_supervisor)

KeyError: 'A'